# 🧠 Ryth — End-to-End Training on Kaggle (T4)

Train Ryth's **30M** coding model completely from scratch, using only the public
[Ryth](https://github.com/RAJ-af/Ryth) library — **no core changes**.

This notebook runs the full pipeline in order:

1. **Environment** — install/locate Ryth, pick device + dtype (fp16 on T4)
2. **Corpus** — build a synthetic code corpus (or point at your own dataset)
3. **Tokenizer** — train the scratch byte-level BPE tokenizer
4. **RDS dataset** — run the Ryth Data Engine → memory-mapped binary shards
5. **Model** — instantiate the 30M transformer (`ryth_30m` preset)
6. **Training** — a short **smoke run** with the Training Engine
7. **Checkpoints & resume** — save, then resume and continue
8. **Generate** — sample code from the trained checkpoint

### How to run on Kaggle
- **Settings → Accelerator → GPU T4 ×2** (or ×1). The notebook auto-detects it.
- To install Ryth from GitHub, enable **Settings → Internet → On**. If internet is
  off, attach the Ryth repo as a *Kaggle dataset* and this notebook will find it.
- Defaults run a fast, self-contained **smoke test** (synthetic data). Flip
  `SMOKE = False` and point `RAW_DIR` at real permissively-licensed code for a
  real run.

> T4 is a Turing GPU (sm_75) with **no native bf16** — the notebook selects
> **fp16** automatically. Ampere+ GPUs get bf16.

## 1 · Environment — install / locate Ryth

In [ ]:

import importlib, subprocess, sys, os

def have_ryth():
    try:
        importlib.import_module("training"); importlib.import_module("model")
        return True
    except Exception:
        return False

if not have_ryth():
    # (a) attached as a Kaggle dataset?  /kaggle/input/<...>/Ryth
    found = None
    for base in ("/kaggle/input",):
        if os.path.isdir(base):
            for root, dirs, files in os.walk(base):
                if "pyproject.toml" in files and os.path.isdir(os.path.join(root, "training")):
                    found = root; break
    if found:
        print("Found Ryth at", found); sys.path.insert(0, found)
    else:
        # (b) install from GitHub (needs Internet = On)
        print("Installing Ryth from GitHub …")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "git+https://github.com/RAJ-af/Ryth.git"], check=False)

assert have_ryth() or True
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2 · Configuration + device / dtype

In [ ]:

# ---- run knobs ----------------------------------------------------------
SMOKE       = True          # fast, self-contained; set False for a real run
RAW_DIR     = None          # e.g. "/kaggle/input/my-python-code"  (None = synthetic)
PRESET      = "ryth_30m"    # ryth_30m | ryth_125m | ryth_350m | ryth_1b
VOCAB       = 2000 if SMOKE else 16000
SEQ_LEN     = 128  if SMOKE else 512
MICRO_BATCH = 8
GRAD_ACCUM  = 2    if SMOKE else 8
STEPS       = 60   if SMOKE else 5000
WARMUP      = 10   if SMOKE else 200
LR          = 3e-4
FORCE_DTYPE = None          # None = auto; or "bf16" | "fp16" | "fp32"

# ---- working dirs (persisted under /kaggle/working) ---------------------
WORK = "/kaggle/working/ryth_work" if os.path.isdir("/kaggle/working") else "ryth_work"
os.makedirs(WORK, exist_ok=True)
RAW   = RAW_DIR or os.path.join(WORK, "raw_repos")
TOK   = os.path.join(WORK, "tok")
RDS   = os.path.join(WORK, "rds_out")
RUN   = os.path.join(WORK, "runs", "ryth-kaggle")

# ---- device + dtype:  T4 (sm_75) has NO native bf16 -> fp16 -------------
import torch
def pick_device_dtype(force=None):
    if not torch.cuda.is_available():
        return "cpu", (force or "fp32")
    major, minor = torch.cuda.get_device_capability()
    auto = "bf16" if major >= 8 else "fp16"      # T4 -> fp16, A100 -> bf16
    return "cuda", (force or auto)

DEVICE, DTYPE = pick_device_dtype(FORCE_DTYPE)
print(f"device={DEVICE}  dtype={DTYPE}  preset={PRESET}  seq_len={SEQ_LEN}  steps={STEPS}")

## 3 · Corpus

By default we generate a **synthetic** corpus of many small, varied Python files
so the data engine produces plenty of unique chunks — this makes the notebook run
with no attached dataset. For a real run, set `RAW_DIR` above to a folder of
**permissively-licensed** code.

In [ ]:

import glob

_TEMPLATES = [
    "def scale_{i}(x):\n    \"\"\"Multiply x by {i}.\"\"\"\n    return x * {i}\n",
    "def add_{i}(a, b):\n    total = a + b + {i}\n    return total\n",
    "class Box{i}:\n    def __init__(self, v):\n        self.v = v + {i}\n\n    def get(self):\n        return self.v\n",
    "def clamp_{i}(x, lo={i}, hi={hi}):\n    if x < lo:\n        return lo\n    if x > hi:\n        return hi\n    return x\n",
    "def fib_{i}(n):\n    a, b = 0, {i}\n    for _ in range(n):\n        a, b = b, a + b\n    return a\n",
    "def filter_{i}(items):\n    return [it for it in items if it % {i} == 0]\n",
]

def make_synthetic_corpus(root, n_files=120):
    os.makedirs(root, exist_ok=True)
    for i in range(n_files):
        tmpl = _TEMPLATES[i % len(_TEMPLATES)]
        parts = [tmpl.format(i=i*7 + k*13 + 1, hi=i*7 + k*13 + 51) for k in range(4)]
        body = f'"""Synthetic module {i}."""\n\n' + "\n".join(parts)
        d = os.path.join(root, f"pkg_{i//20}"); os.makedirs(d, exist_ok=True)
        with open(os.path.join(d, f"mod_{i}.py"), "w") as f:
            f.write(body)
    try:
        from tests.sample_data import build_sample; build_sample(root)
    except Exception:
        pass

if RAW_DIR is None:
    make_synthetic_corpus(RAW)
n_py = len(glob.glob(os.path.join(RAW, "**", "*.py"), recursive=True))
print(f"{n_py} python files under {RAW}")

## 4 · Tokenizer — scratch byte-level BPE

In [ ]:

from dataset import load_bpe_tokenizer
from tokenizer.train import train_tokenizer, iter_text_files

tok_path = os.path.join(TOK, "tokenizer.json")
if not os.path.exists(tok_path):
    texts = list(iter_text_files(os.path.join(RAW, "**", "*.py")))
    print(f"training tokenizer on {len(texts)} files (target vocab {VOCAB}) …")
    train_tokenizer(texts, vocab_size=VOCAB, out_dir=TOK, verbose=True)

tok = load_bpe_tokenizer(tok_path)
print("tokenizer vocab_size =", tok.vocab_size)

## 5 · RDS dataset — the Ryth Data Engine

Cleans, validates, tokenizes, chunks and packs the corpus into memory-mapped
`uint16` RDS shards with a manifest + checksums.

In [ ]:

from dataset import RDEConfig, RDEPipeline, RDSDataset

if not os.path.exists(os.path.join(RDS, "manifest.json")):
    cfg = RDEConfig(seq_len=SEQ_LEN, vocab_size=tok.vocab_size,
                    tokenizer_version=getattr(tok, "version", 1),
                    shard_max_bytes=256 * 1024**2)
    RDEPipeline(tok, cfg).run(RAW, RDS, verbose=True)

ds = RDSDataset(RDS)
n_val   = max(1, int(len(ds) * 0.02))
n_train = len(ds) - n_val
print(f"chunks={len(ds)}  seq_len={ds.seq_len}  (~{n_train} train / {n_val} val)")
assert n_train >= MICRO_BATCH, (
    f"only ~{n_train} train chunks but MICRO_BATCH={MICRO_BATCH}; "
    "lower MICRO_BATCH or use a bigger corpus (DataLoader drop_last would starve).")

## 6 · Model — the 30M transformer

In [ ]:

from model import RythConfig, RythForCausalLM, format_metrics, model_metrics

mcfg = getattr(RythConfig, PRESET)(vocab_size=tok.vocab_size)
if SEQ_LEN > mcfg.max_seq_len:
    mcfg.max_seq_len = SEQ_LEN
model = RythForCausalLM(mcfg)
print(f"{PRESET}: {model.num_params()/1e6:.1f}M params | d_model={mcfg.d_model} "
      f"n_layers={mcfg.n_layers} n_heads={mcfg.n_heads} n_kv={mcfg.n_kv_heads}")
try:
    print(format_metrics(model_metrics(mcfg)))
except Exception:
    pass

## 7 · Train — short smoke run

We hand the model we just built to the `Trainer` (it also accepts `model=None` to
build one from the preset). AdamW + warmup-cosine + mixed precision + gradient
accumulation, with validation + perplexity and checkpointing.

In [ ]:

from training import TrainConfig, Trainer

train_cfg = TrainConfig(
    data_dir=RDS, model_preset=PRESET, seq_len=SEQ_LEN,
    lr=LR, warmup_steps=WARMUP, max_steps=STEPS,
    micro_batch_size=MICRO_BATCH, grad_accum_steps=GRAD_ACCUM,
    dtype=DTYPE, out_dir=RUN,
    eval_every=max(10, STEPS // 3), eval_steps=5, log_every=5,
    save_every=max(10, STEPS // 2), num_workers=2, device=DEVICE,
    run_name="kaggle-smoke")

print(f"effective batch = {train_cfg.effective_batch} | tokens/step = {train_cfg.tokens_per_step}\n")
trainer = Trainer(train_cfg, model=model)
best_val = trainer.train()
print("best_val =", best_val)

## 8 · Checkpoints

In [ ]:

import glob
for c in sorted(glob.glob(os.path.join(RUN, "*.pt"))):
    print(f"{os.path.basename(c):20s} {os.path.getsize(c)/1e6:7.1f} MB")
# best.pt = best validation | final.pt = last step | latest.pt = auto-resume | step_*.pt = rotated

## 9 · Resume from checkpoint

`resume="latest"` rebuilds the model from the preset and restores model +
optimizer + step + RNG from `latest.pt`, then continues training.

In [ ]:

resume_cfg = TrainConfig(
    data_dir=RDS, model_preset=PRESET, seq_len=SEQ_LEN,
    lr=LR, warmup_steps=WARMUP, max_steps=STEPS + 20,
    micro_batch_size=MICRO_BATCH, grad_accum_steps=GRAD_ACCUM,
    dtype=DTYPE, out_dir=RUN,
    eval_every=max(10, STEPS // 3), eval_steps=5, log_every=5,
    save_every=max(10, STEPS // 2), num_workers=2, device=DEVICE,
    resume="latest", run_name="kaggle-resume")

resumed = Trainer(resume_cfg)             # model=None -> built from preset, weights loaded
print("resumed at step:", resumed.step)
resumed.train()

## 10 · Generate code

Load the best checkpoint into a fresh model and sample. (With a short smoke run on
tiny synthetic data the output won't be meaningful — it proves the generate path.
A real corpus + more steps produces real code.)

In [ ]:

from model import generate

best = os.path.join(RUN, "best.pt")
ckpt = best if os.path.exists(best) else os.path.join(RUN, "final.pt")

gen = getattr(RythConfig, PRESET)(vocab_size=tok.vocab_size)
if SEQ_LEN > gen.max_seq_len:
    gen.max_seq_len = SEQ_LEN
gmodel = RythForCausalLM(gen)
state = torch.load(ckpt, map_location=DEVICE, weights_only=False)
gmodel.load_state_dict(state["model"]); gmodel.to(DEVICE).eval()

prompt = "def add(a, b):\n"
ids = tok.encode(prompt) or [getattr(tok, "bos_id", 0)]
x = torch.tensor([ids[-SEQ_LEN:]], dtype=torch.long, device=DEVICE)
out = generate(gmodel, x, max_new_tokens=60, temperature=0.8, top_k=40,
               eos_id=getattr(tok, "eos_id", None))
print(tok.decode(out[0].tolist()))

## ✅ Done — scaling to a real run

You just ran the whole stack: **corpus → tokenizer → RDS → 30M model → training →
checkpoint → resume → generation**, on a T4, without touching the core library.

To train for real:
- Set `SMOKE = False` and point `RAW_DIR` at a large folder of permissively-licensed
  code (attach it as a Kaggle dataset).
- Raise `VOCAB` (e.g. 16k–32k), `SEQ_LEN` (512–1024), and `STEPS` (tens of thousands).
- Kaggle sessions are time-limited — rely on `resume="latest"` to continue across
  sessions (checkpoints persist in `/kaggle/working`).
- For larger models switch `PRESET` to `ryth_125m` / `ryth_350m` and enable
  `grad_checkpointing=True` in `TrainConfig` if you hit memory limits.

Equivalent one-command script: `python scripts/kaggle_train.py --smoke`.